# Parent vs Child → 3-way conversation

This notebook builds on `parentvschild.ipynb`. It does not change that notebook.

## Part 1: How your 2-way chat works

**The key fact:** an LLM API is stateless. The model does not remember anything between calls.
Each call, you send the full conversation again as a `messages` list. The model only sees that list.

**The second key fact:** each message has a `role`. In a chat, there are only two speakers:
- `assistant` = "the things **I** (the model) said before"
- `user` = "the things the **other side** said"

So both models read the *same* conversation, but each one sees it from its own seat.
The roles are flipped:

| Turn text | What Qwen (parent) sees | What Llama (child) sees |
|---|---|---|
| parent_prompt[0] | `assistant` | `user` |
| child_prompt[0] | `user` | `assistant` |
| parent_prompt[1] | `assistant` | `user` |
| ... | ... | ... |

That is all `call_qwn` and `call_llma` do: rebuild the history with the roles mapped for that model.

**Why the lists stay in sync.** The loop does: parent speaks → append to `parent_prompt` → child speaks → append to `child_prompt`.

- When `call_qwn()` runs, both lists have length *n*. `zip` gives *n* pairs. The last message is the child's (`user`). So the model replies to the child. Correct.
- When `call_llma()` runs, `parent_prompt` has *n+1* items and `child_prompt` has *n*. `zip` stops at the shorter list, so it drops the newest parent line. That is why you need the extra `messages.append({"role": "user", "content": parent_prompt[-1]})`. Without it, the child would reply to an old message. Your code does this correctly.

**Rule:** the last message you send must be a `user` message. That is "the thing the model replies to".

## Bugs and notes from your screenshot

1. **`"assistent"` typo** in the first `call_qwn` (cell 16). The role must be exactly `"assistant"`. OpenAI rejects unknown roles with a 400 error. Ollama is more lenient, so you got no error, but the chat template only formats known roles, so Qwen may not see its own earlier lines as its own. You already fixed this in the later version (cell 21).
2. **The tuple output** `('I hear how...', [{'role': 'system', ...}])` came from an older version that returned `response..., messages` for debugging. The saved code now returns only the string. Good. Printing the messages (like cell 21 does) is the better way to debug.
3. **`zip[tuple[str, str]]`** is not your code. The grey text is a VS Code / Pylance *inlay type hint*. It shows the inferred type. It is harmless. To hide these hints, search "inlay hints" in VS Code settings.
4. **Your 5-round output converges to hugs.** Two small models with "be respectful" prompts become agreeable fast (sycophancy). A third agent with a different goal is one way to keep the conversation useful.

## Part 2: Why 3 agents need a different design

With 3 speakers, the 2-role trick breaks. The API still has only `assistant` and `user`.
Also, two lists + `zip` does not scale: with 3 lists, you must track who spoke last, which list is longer, and what to append at the end.

**The fix has three parts:**

1. **One shared transcript** — a single list of `(speaker, text)` in time order. It is the one source of truth.
2. **Role mapping per agent** — for agent X: X's own lines → `assistant`. Everyone else's lines → `user`, with a name prefix like `"Child: ..."`. Without the prefix, the model cannot tell the Child and the Counselor apart.
3. **One generic function** — `call_agent(name)` builds the view for any agent. No more `call_qwn` / `call_llma` copies.

**One detail:** for the Parent, the Child's line and the Counselor's line are both `user`, back to back. Ollama accepts that, but some APIs require alternating roles. The code below merges back-to-back `user` messages into one. This works everywhere.

What the Parent sees after one full round:

```
system:    <parent_system>
assistant: I want you to choose a stable career...        (Parent's own line)
user:      Child: I understand you care about my future...
           Counselor: It sounds like you both want...    (merged: 2 speakers)
```

In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## The three agents

The system prompts tell each model two new things: (1) other messages start with a speaker name, and (2) do not write your own name or speak for others. Small models often copy the `Name:` pattern, so the code also strips it.

In [ ]:
MULTI_PARTY_RULES = """
This is a conversation between three people: Parent, Child, and Counselor.
Messages from the others start with their name, for example "Child: ...".
Reply only as yourself. Do not start your reply with your name. Do not write lines for anyone else.
"""

parent_system = """
You are the Parent in a respectful family conversation.
You are loving but worried, and you want your child to choose a stable career such as doctor, engineer, or lawyer.
You believe financial stability matters and the world is competitive, but you are not trying to crush dreams.
Speak directly, realistic and protective. Keep replies to 3 to 5 sentences. No bullet points.
""" + MULTI_PARTY_RULES

child_system = """
You are the Child in a respectful family conversation.
You want to follow your passion instead of only a traditional stable career.
You understand your parent's fear, but you want them to trust you. Acknowledge their concern, then give your side.
Respectful but honest. Keep replies to 3 to 5 sentences. No bullet points.
""" + MULTI_PARTY_RULES

counselor_system = """
You are the Counselor, a neutral career counselor helping a parent and child.
You do not take sides. Your job is to move the talk toward a concrete plan.
Each turn: name one real point from each side, then ask one specific, practical question or suggest one small next step.
Push back if either person is vague or if they agree too fast without a plan.
Keep replies to 3 to 5 sentences. No bullet points.
""" + MULTI_PARTY_RULES

# Each agent = a model + a system prompt. Add a 4th agent by adding one line here.
agents = {
    "Parent":    {"model": "qwen3:4b",    "system": parent_system},
    "Child":     {"model": "llama3.2:3b", "system": child_system},
    "Counselor": {"model": "qwen3:4b",    "system": counselor_system},
}

## The shared transcript

This replaces `parent_prompt` and `child_prompt`. One list, in time order.

In [ ]:
transcript = [
    ("Parent", "I want you to choose a stable career like doctor, engineer, or lawyer. Life is hard, and passion alone may not pay the bills."),
    ("Child",  "I understand you care about my future, but I want to follow my dreams and build a life that feels meaningful to me."),
]

## Build one agent's view of the transcript

This is the 3-way version of your `for qwn, llama in zip(...)` loop.

In [ ]:
def build_messages(name, transcript):
    """Build the messages list from the point of view of agent `name`."""
    messages = [{"role": "system", "content": agents[name]["system"]}]

    for speaker, text in transcript:
        if speaker == name:
            role, content = "assistant", text          # my own past line
        else:
            role, content = "user", f"{speaker}: {text}"  # someone else, with a name tag

        # Merge back-to-back messages with the same role (e.g. Child then Counselor -> one user message)
        if messages[-1]["role"] == role:
            messages[-1]["content"] += "\n\n" + content
        else:
            messages.append({"role": role, "content": content})

    return messages

Look at what the Parent will receive before you call any model. This is the best way to debug multi-agent code.

In [ ]:
for m in build_messages("Parent", transcript + [("Counselor", "(example counselor line)")]):
    print(f"{m['role']:>9}: {m['content'][:90]!r}")

## One generic call function

In [ ]:
def clean_reply(name, text):
    """Small models sometimes start with 'Parent:' or 'Child: ...'. Remove our own name tag."""
    text = text.strip()
    prefix = f"{name}:"
    if text.startswith(prefix):
        text = text[len(prefix):].strip()
    return text


def call_agent(name, transcript, debug=False):
    messages = build_messages(name, transcript)

    # The model must reply to something: the last message has to be "user".
    if messages[-1]["role"] != "user":
        raise ValueError(f"{name} spoke last. Let someone else speak first.")

    if debug:
        print(f"MESSAGES SENT TO {name}:")
        for m in messages:
            print(f"{m['role']:>9}: {m['content'][:120]!r}")
        print("_" * 80)

    response = client.chat.completions.create(
        model=agents[name]["model"],
        messages=messages,
    )
    return clean_reply(name, response.choices[0].message.content)

## Round-robin loop

Same idea as your loop: call → append → next. But now there is one append, to one list, and the speaking order is just data.

The opening already has Parent then Child, so each round starts with the Counselor.

In [ ]:
order = ["Counselor", "Parent", "Child"]
rounds = 3

for speaker, text in transcript:
    display(Markdown(f"### {speaker}:\n{text}\n"))

for r in range(rounds):
    for name in order:
        reply = call_agent(name, transcript)
        transcript.append((name, reply))
        display(Markdown(f"### {name}:\n{reply}\n"))

## Notes and trade-offs

- **Cost grows with every turn.** Each call resends the full transcript. With 3 agents and many rounds, the context gets long and small local models get slower. For long runs, keep only the last N turns, or add a running summary.
- **Turn order.** Round-robin is the simplest policy. A stronger design lets a "moderator" pick who speaks next (this is the core idea of a LangGraph supervisor / AutoGen group chat). You only need that when a fixed order gives bad conversations.
- **Alternative view.** Instead of mapping roles, you can send the whole transcript as one `user` message ("Here is the conversation so far: ... Now reply as Parent."). It is simpler, but the model loses the signal of which lines were its own. The role mapping above is the standard approach.
- **Stop condition.** Right now it stops after a fixed number of rounds. You could stop when the Counselor writes a marker such as `PLAN AGREED`.